### Download the data from kaggle

In [ ]:
# Setup Kaggle API Credentials

import os
import json

# Create the .kaggle directory if it doesn't exist
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# You'll need to manually copy your kaggle.json file to ~/.kaggle/
# Or you can upload it and move it programmatically
print(f"Place your kaggle.json file in: {kaggle_dir}")

Place your kaggle.json file in: /Users/pasindumalinda/.kaggle


In [5]:
# Create Target Directory and Download Dataset

import os
import shutil

# Define your target directory
target_dir = "/Volumes/KODAK/folder 02/Skin Cancer detection model/data/Raw Data"

# Create the directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# Change to the target directory
os.chdir(target_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Target directory exists: {os.path.exists(target_dir)}")

Current working directory: /Volumes/KODAK/folder 02/Skin Cancer detection model/data/Raw Data
Target directory exists: True


In [12]:
import json
import zipfile
from pathlib import Path
import requests
from kaggle.api.kaggle_api_extended import KaggleApi
import logging
from tqdm import tqdm
import shutil

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('kaggle_downloader.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class KaggleDatasetDownloader:
    def __init__(self, output_dir: str):
        """
        Initialize the Kaggle dataset downloader.
        
        Args:
            output_dir (str): Directory where the dataset will be saved.
        """
        self.output_dir = Path(output_dir)
        self.api = KaggleApi()
        
        # Set custom kaggle.json location if needed
        if not (Path.home() / '.kaggle' / 'kaggle.json').exists():
            custom_kaggle_path = '/Volumes/KODAK/folder 02/Skin Cancer detection model/._kaggle.json'
            if Path(custom_kaggle_path).exists():
                os.environ['KAGGLE_CONFIG_DIR'] = str(Path(custom_kaggle_path).parent)
        
        # Validate and create output directory
        self._prepare_output_directory()
        
    def _prepare_output_directory(self) -> None:
        """Ensure the output directory exists and is writable."""
        try:
            self.output_dir.mkdir(parents=True, exist_ok=True)
            # Test write permission
            test_file = self.output_dir / '.permission_test'
            test_file.touch()
            test_file.unlink()
        except Exception as e:
            logger.error(f"Failed to prepare output directory: {e}")
            raise
            
    def _validate_kaggle_credentials(self) -> bool:
        """Check if Kaggle credentials are properly configured."""
        try:
            kaggle_dir = Path(os.environ.get('KAGGLE_CONFIG_DIR', Path.home() / '.kaggle'))
            kaggle_json = kaggle_dir / 'kaggle.json'
            
            if not kaggle_json.exists():
                logger.error(f"Kaggle credentials not found at {kaggle_json}. Please ensure kaggle.json exists.")
                return False
                
            with open(kaggle_json) as f:
                json.load(f)  # Validate JSON
            return True
        except Exception as e:
            logger.error(f"Invalid kaggle.json file: {e}")
            return False
            
    def _download_with_progress(self, dataset_name: str, destination: Path) -> bool:
        """Download dataset with progress bar."""
        try:
            # Get the download URL
            dataset_files = self.api.dataset_list_files(dataset_name).files
            if not dataset_files:
                logger.error("No files found in dataset")
                return False
                
            # Create progress bar
            with tqdm(unit='B', unit_scale=True, unit_divisor=1024, miniters=1) as pbar:
                def update_progress(block_num, block_size, total_size):
                    if pbar.total != total_size:
                        pbar.total = total_size
                    pbar.update(block_size)
                
                # Download each file in the dataset
                for file in dataset_files:
                    file_path = destination / file.name
                    logger.info(f"Downloading {file.name}...")
                    
                    self.api.dataset_download_file(
                        dataset=dataset_name,
                        file_name=file.name,
                        path=destination,
                        force=True,
                        quiet=True
                    )
                    
                    # The API doesn't provide direct progress, so we simulate it
                    temp_file = destination / file.name
                    if temp_file.exists():
                        temp_file.rename(file_path)
                        pbar.total = os.path.getsize(file_path)
                        pbar.update(os.path.getsize(file_path))
            
            return True
        except Exception as e:
            logger.error(f"Download failed: {e}")
            return False
            
    def download_dataset(self, dataset_name: str, unzip: bool = True, delete_zip: bool = True) -> bool:
        """
        Download a dataset from Kaggle.
        
        Args:
            dataset_name (str): Kaggle dataset identifier in format 'owner/dataset-name'
            unzip (bool): Whether to unzip the downloaded file
            delete_zip (bool): Whether to delete the zip file after extraction
            
        Returns:
            bool: True if download and processing succeeded, False otherwise
        """
        if not self._validate_kaggle_credentials():
            return False
            
        try:
            logger.info(f"Initializing Kaggle API connection...")
            self.api.authenticate()
            
            logger.info(f"Downloading dataset: {dataset_name}")
            
            # Download with progress tracking
            zip_path = self.output_dir / f"{dataset_name.replace('/', '_')}.zip"
            
            # First try the standard download method
            try:
                self.api.dataset_download_files(
                    dataset=dataset_name,
                    path=self.output_dir,
                    quiet=False,  # Let Kaggle show its progress
                    force=True,
                    unzip=False
                )
                
                # Rename the downloaded file to a consistent format
                temp_zip = self.output_dir / f"{dataset_name.split('/')[1]}.zip"
                if temp_zip.exists():
                    temp_zip.rename(zip_path)
            except Exception as e:
                logger.warning(f"Standard download failed, trying alternative method: {e}")
                if not self._download_with_progress(dataset_name, self.output_dir):
                    raise RuntimeError("Both download methods failed")
                
            if not zip_path.exists():
                # Check if files were downloaded without zip
                contents = list(self.output_dir.glob('*'))
                if contents:
                    logger.info(f"Files downloaded directly without zip: {contents}")
                    return True
                raise FileNotFoundError(f"Downloaded files not found at {self.output_dir}")
                
            logger.info(f"Successfully downloaded dataset to {zip_path}")
            
            if unzip and zip_path.exists():
                self._unzip_file(zip_path, delete_zip)
                
            return True
            
        except requests.exceptions.HTTPError as e:
            logger.error(f"HTTP Error occurred: {e}")
            if e.response.status_code == 403:
                logger.error("Authentication failed. Please check your Kaggle API token.")
            elif e.response.status_code == 404:
                logger.error("Dataset not found. Please check the dataset name.")
        except Exception as e:
            logger.error(f"An error occurred while downloading dataset: {e}")
            
        return False
        
    def _unzip_file(self, zip_path: Path, delete_zip: bool = True) -> None:
        """
        Unzip a downloaded dataset.
        
        Args:
            zip_path (Path): Path to the zip file
            delete_zip (bool): Whether to delete the zip file after extraction
        """
        try:
            logger.info(f"Extracting {zip_path.name}...")
            
            # Get total size for progress bar
            total_size = sum(f.file_size for f in zipfile.ZipFile(zip_path).infolist())
            
            with tqdm(total=total_size, unit='B', unit_scale=True, desc="Extracting") as pbar:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    for file in zip_ref.infolist():
                        try:
                            zip_ref.extract(file, self.output_dir)
                            pbar.update(file.file_size)
                        except Exception as e:
                            logger.warning(f"Failed to extract {file.filename}: {e}")
                            continue
                        
            logger.info(f"Extraction complete to {self.output_dir}")
            
            if delete_zip:
                zip_path.unlink()
                logger.info(f"Deleted zip file: {zip_path.name}")
                
        except zipfile.BadZipFile:
            logger.error(f"File is not a zip file or is corrupted: {zip_path}")
        except Exception as e:
            logger.error(f"Error during extraction: {e}")
            

def main():
    # Configuration
    DATASET_NAME = "kmader/skin-cancer-mnist-ham10000"
    OUTPUT_DIR = "/Volumes/KODAK/folder 02/Skin Cancer detection model/data/Raw Data/skin-cancer-mnist-ham10000"
    
    try:
        downloader = KaggleDatasetDownloader(OUTPUT_DIR)
        success = downloader.download_dataset(DATASET_NAME)
        
        if success:
            logger.info("Dataset download and processing completed successfully!")
        else:
            logger.error("Dataset download failed.")
            exit(1)
            
    except Exception as e:
        logger.error(f"Fatal error in main execution: {e}")
        exit(1)
        

if __name__ == "__main__":
    main()

2025-07-08 18:22:35,222 - INFO - Initializing Kaggle API connection...
2025-07-08 18:22:35,223 - INFO - Downloading dataset: kmader/skin-cancer-mnist-ham10000


Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000


100%|██████████| 5.20G/5.20G [01:00<00:00, 92.8MB/s]
2025-07-08 20:47:49,766 - INFO - Successfully downloaded dataset to /Volumes/KODAK/folder 02/Skin Cancer detection model/data/Raw Data/skin-cancer-mnist-ham10000/kmader_skin-cancer-mnist-ham10000.zip
2025-07-08 20:47:49,774 - INFO - Extracting kmader_skin-cancer-mnist-ham10000.zip...


Extracting: 100%|██████████| 5.67G/5.67G [01:24<00:00, 67.0MB/s]
2025-07-08 20:49:14,565 - INFO - Extraction complete to /Volumes/KODAK/folder 02/Skin Cancer detection model/data/Raw Data/skin-cancer-mnist-ham10000
2025-07-08 20:49:14,618 - INFO - Deleted zip file: kmader_skin-cancer-mnist-ham10000.zip
2025-07-08 20:49:14,623 - INFO - Dataset download and processing completed successfully!
